In [ ]:
import json
import logging
import shutil
from datetime import date, datetime
from pathlib import Path

import requests
import pandas as pd

In [ ]:
# Rutas
OUTPUT_DIR = Path(
    "C:/Users/Usuario/OneDrive - Global Green Growth Institute"
    "/Documentos/2025/Outputs/Output4/Indicadores/Territorios"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUTPUT_DIR / "territorios.log"
logging.basicConfig(
    filename=LOG_FILE, level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S", encoding="utf-8",
)
log = logging.getLogger("territorios")

# Hub ArcGIS: descarga directa GeoJSON sin autenticacion
HUB_DL = "https://hub.arcgis.com/api/download/v1/items/{item_id}/geojson?redirect=true&layers=0"
CAPAS = [
    {"id": "resguardos_indigenas",    "titulo": "Resguardo Indigena Formalizado",
     "item_id": "8944116ccfd34a7189c4bc44b8e19186"},
    {"id": "consejos_comunitarios",   "titulo": "Consejo Comunitario Titulado",
     "item_id": "abf2f9f6727b4073902c1f57c280d5dc"},
    {"id": "zonas_reserva_campesina", "titulo": "Zonas de Reserva Campesina Constituida",
     "item_id": "0eca5beb8afe43708622fdd7646cd577"},
    {"id": "territorios_campesinos",  "titulo": "Territorios Campesinos Agroalimentarios",
     "item_id": "df0e553ffbab4a609e1fd43c437e216b"},
]

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 Python/requests"})

In [ ]:
def descargar_geojson(item_id, titulo):
    """Descarga el GeoJSON completo desde ArcGIS Hub (stream)."""
    url = HUB_DL.format(item_id=item_id)
    log.info(f"GET {url}")
    r = SESSION.get(url, stream=True, timeout=300)
    r.raise_for_status()
    chunks = []
    total  = 0
    for chunk in r.iter_content(chunk_size=1024 * 256):
        chunks.append(chunk)
        total += len(chunk)
        print(f"  descargado: {total / 1024**2:.1f} MB", end="", flush=True)
    print()
    data = json.loads(b"".join(chunks))
    features = data.get("features", [])
    log.info(f"[{titulo}] {len(features)} features descargados")
    return features


def guardar(features, clave):
    fecha  = date.today().isoformat()
    p_json = OUTPUT_DIR / f"{clave}.geojson"
    p_xlsx = OUTPUT_DIR / f"{clave}.xlsx"
    geojson = {"type": "FeatureCollection", "features": features,
               "_meta": {"fecha_descarga": fecha, "total": len(features)}}
    tmp = OUTPUT_DIR / f"_tmp_{clave}.geojson"
    tmp.write_text(json.dumps(geojson, ensure_ascii=False, indent=2), encoding="utf-8")
    shutil.move(str(tmp), str(p_json))
    rows  = [{"fecha_descarga": fecha, **f.get("properties", {})} for f in features]
    df    = pd.DataFrame(rows)
    tmp_x = OUTPUT_DIR / f"_tmp_{clave}.xlsx"
    df.to_excel(tmp_x, index=False, engine="openpyxl", sheet_name=clave[:31])
    shutil.move(str(tmp_x), str(p_xlsx))
    log.info(f"Guardado {clave}: {len(features)} features")
    print(f"  -> {p_json.name}  |  {p_xlsx.name}  ({len(df)} filas)")

In [ ]:
inicio = datetime.now()
log.info("=" * 60)
log.info(f"INICIO  {inicio:%Y-%m-%d %H:%M:%S}")
log.info("=" * 60)
print(f"Descargando territorios ANT - {inicio:%Y-%m-%d %H:%M}")

errores = []
for capa in CAPAS:
    print("-" * 50)
    print(capa["titulo"])
    try:
        features = descargar_geojson(capa["item_id"], capa["titulo"])
        if not features:
            print("  [AVISO] Sin registros.")
            log.warning(f"Sin features: {capa['titulo']}")
        else:
            guardar(features, capa["id"])
    except Exception as exc:
        errores.append(capa["titulo"])
        log.error(f"ERROR {exc}", exc_info=True)
        print(f"  [ERROR] {exc}")

elapsed = (datetime.now() - inicio).seconds
ok = len(CAPAS) - len(errores)
print(f"Completado: {ok}/{len(CAPAS)} capas  ({elapsed}s)")
if errores:
    print(f"Con error: {errores}")
print(f"Salida: {OUTPUT_DIR}")
log.info(f"FIN  {ok}/{len(CAPAS)} OK  {elapsed}s")
log.info("=" * 60)

In [ ]:
# Spatial join: territorios ANT x municipios de cartera -> territorios_municipal.csv
import geopandas as gpd
import pandas as pd
from pathlib import Path

WEB_DATA = Path(
    "C:/Users/Usuario/OneDrive - Global Green Growth Institute"
    "/Documentos/2025/Outputs/Output1/Stress Test/3.Data/Scripts Python"
    "/webpage_climate/data"
)

# Municipios de cartera con geometria
muni = gpd.read_file(WEB_DATA / "municipios_colombia_geo.gpkg")
muni["COD_MPIO"] = muni["COD_MPIO"].astype(float).astype(int)

# Coordenadas de referencia (centroide oficial)
coords = pd.read_csv(WEB_DATA / "datos_municipios.csv",
    usecols=["COD_MPIO","NOM_MPIO","NOM_DPTO","LATITUD","LONGITUD"]).drop_duplicates("COD_MPIO")

# Municipios de la cartera
cartera = pd.read_csv(WEB_DATA / "cartera_municipios_codigos.csv")[["COD_MPIO"]]

# Filtrar geometrias a la cartera
muni_c = muni[muni["COD_MPIO"].isin(cartera["COD_MPIO"])][["COD_MPIO","geometry"]].copy()

# Configuracion de capas
TERR = [
    {"clave": "resguardos_indigenas",    "col": "resguardo"},
    {"clave": "consejos_comunitarios",   "col": "consejo"},
    {"clave": "zonas_reserva_campesina", "col": "zrc"},
    {"clave": "territorios_campesinos",  "col": "tca"},
]

# Base: cartera + coords
base = cartera.merge(coords, on="COD_MPIO", how="left")

for cfg in TERR:
    path_gj = OUTPUT_DIR / f"{cfg['clave']}.geojson"
    if not path_gj.exists():
        print(f"  [OMITIDO] {path_gj.name} no encontrado")
        for sfx in ["tiene","nombre","n"]:
            base[f"{sfx}_{cfg['col']}"] = 0 if sfx != "nombre" else ""
        continue
    gdf = gpd.read_file(path_gj).to_crs(muni_c.crs)
    joined = gpd.sjoin(muni_c, gdf[["NOMBRE","geometry"]], how="left", predicate="intersects")
    agg_name = joined.groupby("COD_MPIO")["NOMBRE"].apply(lambda x: "|".join(x.dropna().unique()))
    agg_n    = joined.groupby("COD_MPIO")["NOMBRE"].count()
    base[f"tiene_{cfg['col']}"]  = base["COD_MPIO"].isin(agg_name[agg_name != ""].index).astype(int)
    base[f"nombre_{cfg['col']}"] = base["COD_MPIO"].map(agg_name).fillna("")
    base[f"n_{cfg['col']}s"]     = base["COD_MPIO"].map(agg_n).fillna(0).astype(int)
    n = base[f"tiene_{cfg['col']}"].sum()
    print(f"  {cfg['clave']:35s}  {n:>4} municipios con traslape")

# Exportar
out = WEB_DATA / "territorios_municipal.csv"
base.to_csv(out, index=False, encoding="utf-8-sig")
print(f"
Exportado: {out}  ({len(base)} municipios)")